In [1]:
print("Kernel works")

Kernel works


In [7]:
import pandas as pd
from rdkit import Chem

df = pd.read_csv ('/home/susan/mof-co2-adsorption/data/processed/df_chem.csv')
print(df.columns)
print("Dataset shape:", df.shape)
print("Missing MOFID:", df["mofid"].isna().sum())
print("Unique MOFID:", df["mofid"].nunique())


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')
Dataset shape: (27706, 11)
Missing MOFID: 0
Unique MOFID: 24958


In [10]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


---
we want to answer three questions:

- How many extra occurrences are caused by repeated MOFIDs?
- How many different MOFID strings appear more than once?
- How many times can one MOFID occur?

Step 1 — Count occurrence of each MOFID.


In [17]:
mofid_counts = df["mofid"].value_counts()
# Keep only MOFIDs that appear more than once
repeated_mofids = mofid_counts[mofid_counts > 1]

print("Total rows:", len(df))
print("Unique MOFIDs:", df["mofid"].nunique())
print("Repeated occurrences:", df["mofid"].duplicated().sum())

print(mofid_counts.head())


Total rows: 27706
Unique MOFIDs: 24958
Repeated occurrences: 2748
mofid
* MOFid-v1.NA.NA                                                                         1259
* MOFid-v1.NA.NAno_mof                                                                    495
N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERROR.cat0                                     39
N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERROR.cat0                                     32
[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21
Name: count, dtype: int64


| Output                                | Meaning                                                                                                                                             |
| ------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Total rows: 27,706**                |  dataset contains 27,706 MOF records with a non-null value in the `mofid` column.                                                               |
| **Unique MOFIDs: 24,958**             | There are 24,958 different `mofid` strings among those 27,706 rows.                                                                                 |
| **Extra repeated occurrences: 2,748** | After keeping the first occurrence of every MOFID, there are 2,748 additional occurrences of already-seen MOFID strings. This is `27,706 − 24,958`. |
| **Unique MOFIDs that repeat: 616**    | There are 616 different MOFID strings that occur at least twice.                                                                                    |
| **Maximum occurrence: 1,259**         | The most frequently occurring MOFID string appears in 1,259 rows.                                                                                   |


Total rows: 27706
Unique MOFIDs: 24958
Extra repeated occurrences: 2748
Unique MOFIDs that repeat: 616 `number of repeated MOFID types`
Maximum occurrence of one MOFID: 1259 `highest frequency of one MOFID type`
mofid

---


`* MOFid-v1.NA.NA    1259`
means this exact string occurs in 1,259 rows. NA indicates that a normal MOFID representation was not available/generated, so these are not 1,259 copies of the same chemical structure.

---
*MOFid-v1.NA.NAno_mof   495*
means this exact status-like string occurs 495 times. Again, this is not a normal chemical MOFID.

---

*N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERRORcat0  39*

means the exact same MOFID string occurs 39 times. Unlike the NA entries, it contains chemical fragments (N#N, linker, Zn) but its MOFID metadata contains ERROR.

---

*N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERRORcat0   32*
is another specific chemical representation occurring 32 times, also carrying an ERROR status.

---
*[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21*
occurs 21 times. This looks different because pcu is a topology designation rather than an NA or ERROR marker.

*The key distinction is:*
27,706 rows ≠ 27,706 unique MOFIDs. There are 24,958 unique MOFID strings, and the repetition is strongly influenced by placeholder/error values such as the 1,259 NA records.

removing both
* MOFid-v1.NA.NA
          ^^^^^^^^^^

* MOFid-v1.NA.NAno_mof
          ^^^^^^^^^^

In [32]:
# removing both
# MOFid-v1.NA.NA
#  MOFid-v1.NA.NAno_mof
df = df[
    ~df["mofid"].str.contains("MOFid-v1.NA", na=False)
].copy()

print("Remaining rows:", len(df))
#Then verify:
print(df.shape)
print(df["mofid"].str.contains("MOFid-v1.NA", na=False).sum())


Remaining rows: 25952
(25952, 11)
0


In [ ]:
# quantify only the ERROR records:
error_count = df["mofid"].str.contains(
    "MOFid-v1.ERROR", na=False
).sum()

print("ERROR-type MOFIDs:", error_count)

# take Take one ERROR MOFID to check it works with rdkit
error_example = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"].iloc[0]

print(error_example)

# extract it :
chemical_smiles = error_example.split(" ")[0]
print(chemical_smiles)

# Test with RDKIT 
mol = Chem.MolFromSmiles(chemical_smiles)
print(mol)

ERROR-type MOFIDs: 1691
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.ERROR
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn]
None
result: MOFID contains chemical text, but RDKit cannot parse it successfully as written.


[13:39:04] Explicit valence for atom # 89 O, 3, is greater than permitted


In [36]:
# Now test all 1,691 ERROR records with RDkit : 
error_mofids = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"]

parsed = 0
failed = 0

for mofid in error_mofids:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
    else:
        parsed += 1

print("Total ERROR records:", len(error_mofids))
print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

[13:40:59] Explicit valence for atom # 89 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 25 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 33 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 23 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 13 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 31 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 101 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 33 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 63 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 51 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 55 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 54 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 51 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom 

Total ERROR records: 1691
Successfully parsed: 1628
Failed to parse: 63


[13:40:59] Explicit valence for atom # 33 C, 5, is greater than permitted
[13:40:59] Explicit valence for atom # 133 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 64 N, 4, is greater than permitted
[13:40:59] Explicit valence for atom # 15 N, 4, is greater than permitted
[13:40:59] Explicit valence for atom # 197 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 19 N, 4, is greater than permitted
[13:40:59] Explicit valence for atom # 4 N, 4, is greater than permitted
[13:40:59] Explicit valence for atom # 3 N, 4, is greater than permitted
[13:40:59] Explicit valence for atom # 137 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 89 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 12 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 37 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom # 37 O, 3, is greater than permitted
[13:40:59] Explicit valence for atom 


*27,706 non-null MOFID rows → remove 1,754 NA placeholders → 25,952 rows → 1,691 ERROR-labelled records → 1,628 parse successfully and only 63 fail.*

---




### Can RDKit parse the chemical representation for all 25,952 remaining records?